# WNBA Draft Fit Predictor — Modeling

This notebook builds a model to predict Rookie Impact Score (RIS)
for the 2026 draft class based on NCAA career stats and team context.

Target variable: RIS (Rookie Impact Score)
Training data: 2019-2025 Round 1 rookies
Prediction: 2026 Round 1 draft class

In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load data
rookies_df = pd.read_csv("../data/processed/rookies_clean.csv")
rookie_ncaa_features = pd.read_csv("../data/processed/rookie_ncaa_features.csv")
prospect_features = pd.read_csv("../data/processed/prospect_team_features.csv")
teams_df = pd.read_csv("../data/processed/team_stats_clean.csv")

# Recalculate RIS
ris_features = ['wnba_games', 'wnba_mpg', 'wnba_ppg', 'wnba_rpg', 'wnba_apg', 'wnba_ws40']
scaler = MinMaxScaler()
rookies_scaled = rookies_df.copy()
rookies_scaled[ris_features] = scaler.fit_transform(rookies_df[ris_features])

rookies_df['RIS'] = (
    0.20 * rookies_scaled['wnba_games'] +
    0.20 * rookies_scaled['wnba_mpg'] +
    0.20 * rookies_scaled['wnba_ppg'] +
    0.15 * rookies_scaled['wnba_rpg'] +
    0.10 * rookies_scaled['wnba_apg'] +
    0.15 * rookies_scaled['wnba_ws40']
).round(3)

# Build training set
training_df = rookies_df.merge(rookie_ncaa_features, on='player', how='inner')

print(f"Training set: {training_df.shape}")
print(f"RIS range: {training_df['RIS'].min():.3f} - {training_df['RIS'].max():.3f}")

Training set: (86, 30)
RIS range: 0.064 - 0.809


In [10]:
# Define features for modeling
ncaa_features = ['avg_G', 'avg_MP', 'avg_FG%', 'avg_3P%', 'avg_FT%', 
                 'avg_ORB', 'avg_DRB', 'avg_TRB', 'avg_AST', 'avg_STL', 
                 'avg_BLK', 'avg_TOV', 'avg_PTS', 'avg_eFG%',
                 'seasons_played', 'final_season_pts']

# Target variable
X = training_df[ncaa_features]
y = training_df['RIS']

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nFeature list: {ncaa_features}")

Features: 16
Samples: 86

Feature list: ['avg_G', 'avg_MP', 'avg_FG%', 'avg_3P%', 'avg_FT%', 'avg_ORB', 'avg_DRB', 'avg_TRB', 'avg_AST', 'avg_STL', 'avg_BLK', 'avg_TOV', 'avg_PTS', 'avg_eFG%', 'seasons_played', 'final_season_pts']


In [11]:
from sklearn.model_selection import cross_val_score

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define models
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Evaluate each model using cross validation
print("Model Performance (Cross Validation R²):")
print("-" * 45)

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')
    rmse_scores = cross_val_score(model, X_scaled, y, cv=5, 
                                  scoring='neg_root_mean_squared_error')
    print(f"\n{name}:")
    print(f"  R²:   {scores.mean():.3f} (+/- {scores.std():.3f})")
    print(f"  RMSE: {(-rmse_scores.mean()):.3f} (+/- {rmse_scores.std():.3f})")

Model Performance (Cross Validation R²):
---------------------------------------------

Ridge Regression:
  R²:   0.012 (+/- 0.365)
  RMSE: 0.163 (+/- 0.018)

Random Forest:
  R²:   -0.099 (+/- 0.527)
  RMSE: 0.170 (+/- 0.035)

Gradient Boosting:
  R²:   -0.365 (+/- 0.643)
  RMSE: 0.189 (+/- 0.038)


In [12]:
# Add draft pick as a feature
training_df['draft_pick'] = pd.to_numeric(training_df['draft_pick'], errors='coerce')

ncaa_features_v2 = ncaa_features + ['draft_pick']

X2 = training_df[ncaa_features_v2]
X2_scaled = scaler.fit_transform(X2)

print("Model Performance with Draft Pick Added:")
print("-" * 45)

for name, model in models.items():
    scores = cross_val_score(model, X2_scaled, y, cv=5, scoring='r2')
    rmse_scores = cross_val_score(model, X2_scaled, y, cv=5,
                                  scoring='neg_root_mean_squared_error')
    print(f"\n{name}:")
    print(f"  R²:   {scores.mean():.3f} (+/- {scores.std():.3f})")
    print(f"  RMSE: {(-rmse_scores.mean()):.3f} (+/- {rmse_scores.std():.3f})")

Model Performance with Draft Pick Added:
---------------------------------------------

Ridge Regression:
  R²:   0.172 (+/- 0.469)
  RMSE: 0.145 (+/- 0.022)

Random Forest:
  R²:   0.077 (+/- 0.405)
  RMSE: 0.157 (+/- 0.030)

Gradient Boosting:
  R²:   -0.136 (+/- 0.613)
  RMSE: 0.171 (+/- 0.029)


In [13]:
# Train final Ridge model on all training data
from sklearn.linear_model import Ridge

final_model = Ridge(alpha=1.0)
final_scaler = StandardScaler()

X_final = training_df[ncaa_features_v2]
X_final_scaled = final_scaler.fit_transform(X_final)
final_model.fit(X_final_scaled, y)

# Prepare 2026 prospect features
# Add draft pick for each prospect
draft_picks = {
    "Azzi Fudd": 1,
    "Olivia Miles": 2,
    "Lauren Betts": 4,
    "Gabriela Jaquez": 5,
    "Kiki Rice": 6,
    "Flau'jae Johnson": 8,
    "Angela Dugalic": 9,
    "Raven Johnson": 10,
    "Cotie McMahon": 11,
    "Madina Okot": 13,
    "Taina Mair": 14,
    "Gianna Kneepkens": 15,
}

prospect_features['draft_pick'] = prospect_features['player'].map(draft_picks)

# Select same features
X_prospects = prospect_features[ncaa_features_v2]
X_prospects_scaled = final_scaler.transform(X_prospects)

# Predict
prospect_features['predicted_RIS'] = final_model.predict(X_prospects_scaled).round(3)

print("2026 Predicted Rookie Impact Scores:")
print("-" * 45)
print(prospect_features[['player', 'wnba_team', 'draft_pick', 'predicted_RIS']]
      .sort_values('predicted_RIS', ascending=False)
      .to_string())

2026 Predicted Rookie Impact Scores:
---------------------------------------------
              player           wnba_team  draft_pick  predicted_RIS
7       Lauren Betts  Washington Mystics           4          0.595
8        Madina Okot       Atlanta Dream          13          0.521
6          Kiki Rice       Toronto Tempo           6          0.496
3   Flau'jae Johnson       Seattle Storm           8          0.486
4    Gabriela Jaquez         Chicago Sky           5          0.481
9       Olivia Miles      Minnesota Lynx           2          0.476
1          Azzi Fudd        Dallas Wings           1          0.414
10     Raven Johnson       Indiana Fever          10          0.333
11        Taina Mair       Seattle Storm          14          0.326
5   Gianna Kneepkens     Connecticut Sun          15          0.301
0     Angela Dugalic  Washington Mystics           9          0.300
2      Cotie McMahon  Washington Mystics          11          0.288


In [14]:
# Save predictions
prospect_features.to_csv("../data/processed/predictions_2026.csv", index=False)
print("Saved predictions_2026.csv!")

Saved predictions_2026.csv!
